In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client

In [2]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelGP/ModelGP_RBF.json")
client.get_next_trials(max_trials=1)

{56: {'n_ci': 0.0, 'n_it': 0.43972454147210355}}

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 1000

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.QuasirandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(client.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[14.882969660063692, 13.503850684287865, 15.03148232066246, 14.713285368474065, 15.082092485773938, 14.729057538671404, 14.705013254679953, 14.93368305139767, 14.683765099816227, 14.803642108599787, 14.483874146502737, 15.129475495216152, 14.724349204314827, 14.590712762162617, 15.040218075254563, 15.139606544095393, 14.906103042696891, 14.886465153264853, 14.9148334428082, 14.471359609893625, 15.104407630346813, 14.103973861479442, 15.09767251592386, 14.849586930270723, 14.999445561436179, 14.995504188955143, 14.716846075866206, 15.127131441983222, 15.024213289304377, 15.101219413818235, 14.959328924471022, 15.1270687895903, 14.421682445385063, 14.585037392467205, 14.097010568908624, 15.130432910707977, 15.090531444154575, 15.071588186908414, 14.94739778024591, 14.67862270043131, 14.979720351492745, 14.864639596796675, 15.103260046089016, 14.552545720063325, 14.700315781913847, 15.12900594422391, 14.272287362320014, 14.979966562718609, 15.064330105465794, 15.107221419058817, 14.915425

In [5]:
np.average(y_max_arr)

np.float64(14.816527337819734)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_RBF/DataGenerated/quasirandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)